# SmartHouse AI — Exploratory Data Analysis & Real Estate Market Insights

**System**: SmartHouse AI — Explainable House Price Prediction & Real Estate Analytics System  
**Dataset**: Bengaluru Metropolitan Real Estate Transactions (12,395 cleaned records)  
**Target Column**: `price` (in ₹ Lakhs INR)  

---

## Objectives of this Notebook
1. **Dataset Overview & Schema Inspection**: Audit structural distributions, features, and missing data.
2. **Target Variable Analysis**: Analyze price central tendency, spread, and evaluate log-transformation for regression.
3. **Feature Interactions**: Examine built-up area (`total_sqft`), bedroom count (`bhk`), bathrooms, and area types against price.
4. **Location Intelligence**: Profile locality price benchmarks, prime high-demand corridors, and affordable housing clusters.
5. **Price per Square Foot Dynamics**: Compute unit pricing for market analysis (**strictly isolated to prevent target leakage in ML models**).
6. **Correlation & Multicollinearity**: Quantify linear and non-linear relationships across housing attributes.
7. **Outlier Profiling**: Differentiate genuine luxury properties from potential data anomalies.
8. **Key Findings**: Formulate domain insights and feature engineering hypotheses for Phase 4.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.insert(0, '..')
from src.utils.config import CLEANED_DATA_PATH, VISUALIZATIONS_DIR
from src.data.eda import (
    compute_dataset_overview,
    compute_target_statistics,
    calculate_price_per_sqft,
    compute_numerical_summaries,
    compute_categorical_summaries,
    compute_correlation_matrix,
    compute_location_statistics,
    compute_outlier_summary,
    generate_eda_visualizations
)

sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams['figure.dpi'] = 120
print('SmartHouse AI EDA environment ready.')

## 1. Dataset Loading & High-Level Overview

In [ ]:
df = pd.read_csv(CLEANED_DATA_PATH)
print(f'Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns')
display(df.head())
display(df.describe().T)

## 2. Target Variable Analysis (`price` in ₹ Lakhs)

We evaluate the distribution of the target variable `price` to inspect skewness and test if a log transformation log(1 + y) normalizes the residuals for linear and gradient-boosted models.

In [ ]:
target_stats = compute_target_statistics(df, target_col='price')
for k, v in target_stats.items():
    print(f'{k:20s}: {v}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['price'], bins=40, kde=True, ax=axes[0], color='#2b5c8f')
axes[0].set_title('Raw Price Distribution (INR Lakhs)', fontweight='bold')
axes[0].axvline(df['price'].median(), color='red', linestyle='--', label=f"Median: INR {df['price'].median():.1f}L")
axes[0].legend()

sns.histplot(np.log1p(df['price']), bins=40, kde=True, ax=axes[1], color='#1b9e77')
axes[1].set_title('Log-Transformed Target: log(1 + Price)', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Built-up Area (`total_sqft`) vs Price Relationship

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df[df['total_sqft'] <= 6000], x='total_sqft', y='price', hue='bhk', palette='viridis', alpha=0.6)
plt.title('Total Area (Sq. Ft) vs Price (INR Lakhs)', fontweight='bold')
plt.xlabel('Total Area (Sq. Ft)')
plt.ylabel('Price (INR Lakhs)')
plt.show()

## 4. Room Configuration (`bhk` and `bath`) vs Price Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
bhk_filt = df[df['bhk'].between(1, 6)]
sns.boxplot(data=bhk_filt, x='bhk', y='price', hue='bhk', palette='Blues_r', ax=axes[0], legend=False, showmeans=True)
axes[0].set_title('Price Distribution by BHK', fontweight='bold')
axes[0].set_ylim(0, 400)

bath_filt = df[df['bath'].between(1, 6)]
sns.boxplot(data=bath_filt, x='bath', y='price', hue='bath', palette='crest', ax=axes[1], legend=False, showmeans=True)
axes[1].set_title('Price Distribution by Bathrooms', fontweight='bold')
axes[1].set_ylim(0, 400)
plt.tight_layout()
plt.show()

## 5. Structural Categories: Area Type & Availability Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=df, x='area_type', y='price', hue='area_type', palette='Set2', ax=axes[0], legend=False, showmeans=True)
axes[0].set_title('Price Distribution across Area Types', fontweight='bold')
axes[0].set_ylim(0, 350)

sns.boxplot(data=df, x='availability_status', y='price', hue='availability_status', palette='pastel', ax=axes[1], legend=False, showmeans=True)
axes[1].set_title('Price Comparison: Ready vs Under Construction', fontweight='bold')
axes[1].set_ylim(0, 350)
plt.tight_layout()
plt.show()

## 6. Location Valuation Intelligence & Price per Sq. Ft

> **Important Data Leakage Rule**: `price_per_sqft` is derived from `price` and is strictly calculated for market analysis. It must NEVER be used as a predictive feature.

In [ ]:
df['price_per_sqft'] = calculate_price_per_sqft(df)
loc_stats = compute_location_statistics(df, min_listings=15)

top_expensive_df = pd.DataFrame(loc_stats['top_15_expensive_localities'])
plt.figure(figsize=(12, 6))
sns.barplot(data=top_expensive_df, x='median_price_per_sqft', y='location', hue='location', palette='rocket', legend=False)
plt.title('Top 15 Prime / High-Demand Localities (Median INR / Sq. Ft)', fontweight='bold')
plt.xlabel('Median INR / Sq. Ft')
plt.show()

## 7. Correlation Matrix & Multicollinearity Profiling

In [ ]:
corr = compute_correlation_matrix(df)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Real Estate Attributes', fontweight='bold')
plt.show()

## 8. Outlier Analysis & Summary Diagnostics

In [ ]:
outlier_rep = compute_outlier_summary(df)
display(pd.DataFrame(outlier_rep).T)
print('EDA successfully completed. All visualizations saved to visualizations/eda/.')